# Export for Android

Export the trained detector to LiteRT/TFLite. The export is saved in the Drive project folder so it can be copied into the Android app later. The current float32 export expects [1, 3, 640, 640] RGB input and emits raw [1, 5, 8400] detections with NMS disabled.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/TrayMachi/indonesia-license-plate-model.git"
REPO_DIR = Path("/content/indonesia-license-plate-model")
if not (REPO_DIR / "requirements.txt").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
%pip install -q -r /content/indonesia-license-plate-model/requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/indonesia-license-plate-model")
CHECKPOINT = DRIVE_ROOT / "runs" / "plate-detector" / "weights" / "best.pt"
EXPORT_DIR = DRIVE_ROOT / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
if not CHECKPOINT.exists():
    raise FileNotFoundError(f"Update CHECKPOINT; file not found: {CHECKPOINT}")

In [ ]:
from ultralytics import YOLO

model = YOLO(str(CHECKPOINT))
exported_path = model.export(format="tflite", imgsz=640, int8=False, nms=False)
exported_path = Path(exported_path)
print("Exported model:", exported_path)
print("Input size: 640x640 RGB, float32 [0, 1] by default")

In [ ]:
import shutil

final_path = EXPORT_DIR / exported_path.name
if exported_path.resolve() != final_path.resolve():
    shutil.copy2(exported_path, final_path)
print("Android artifact:", final_path)
print("Inspect the export metadata and tensor shapes before integrating it with LiteRT.")

In [ ]:
import sys

import cv2
import tensorflow as tf

sys.path.insert(0, str(REPO_DIR))
from src.postprocess import decode_yolo11_output, scale_boxes_to_original
from src.preprocess import letterbox, to_model_input

test_images = sorted((DRIVE_ROOT / "dataset_yolo" / "images" / "test").glob("*"))
if not test_images:
    raise FileNotFoundError("No prepared test images found.")
test_image_path = test_images[0]
image_bgr = cv2.imread(str(test_image_path))
if image_bgr is None:
    raise ValueError(f"Could not read test image: {test_image_path}")
image_rgb, ratio, padding = letterbox(image_bgr, (640, 640))
input_tensor = to_model_input(image_rgb)

interpreter = tf.lite.Interpreter(model_path=str(final_path))
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]
if tuple(input_tensor.shape) != tuple(input_detail["shape"]):
    raise ValueError(f"Preprocessing shape {input_tensor.shape} does not match model shape {input_detail['shape']}")
interpreter.set_tensor(input_detail["index"], input_tensor)
interpreter.invoke()
raw_output = interpreter.get_tensor(output_detail["index"])
boxes, scores, class_ids = decode_yolo11_output(raw_output)
boxes = scale_boxes_to_original(boxes, ratio, padding, image_bgr.shape[:2])

print("Test image:", test_image_path.name)
print("LiteRT input:", input_detail["shape"].tolist(), str(input_detail["dtype"]))
print("LiteRT raw output:", raw_output.shape, str(raw_output.dtype))
print("Decoded detections:", len(scores))
for box, score, class_id in zip(boxes, scores, class_ids):
    print({"box_xyxy": box.round(1).tolist(), "score": round(float(score), 4), "class_id": int(class_id)})

reference = model.predict(source=str(test_image_path), conf=0.25, imgsz=640, verbose=False)[0]
print("Ultralytics reference detections:", len(reference.boxes))

The Android client should use the same letterbox parameters as src/preprocess.py, feed NCHW float32 input, then apply decode_yolo11_output and scale_boxes_to_original. Quantized exports can be added later after this float32 baseline is verified.